<a href="https://colab.research.google.com/github/Kshitij8097/UofT_Machine_Learning_3253/blob/main/Auto_theft_(Supervised_with_label)_Kshitij_copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Business Problem**

As a auto-theft insurer that provides (6 months / 1 year) insurance, I want to be able to determine if the location of my customer is high/low risk so that I will be able to determine the appropriate premium.

*I added a time period since a location may not be perpetually low or high risk since the situation of a location may change overtime.


In [ ]:
import numpy as np
import pandas as pd
import requests
import re

# DATA PREPARATION

Load the data

In [ ]:
import requests
import pandas as pd

#original data source https://data.tps.ca/datasets/TorontoPS::auto-theft-open-data/about
file_id = '190IgZ2OhktJOw7K7Fws_yMkWzQVznmOp'
download_url = f'https://drive.google.com/uc?export=download&id={file_id}'

# Download the file
response = requests.get(download_url)
response.raise_for_status() # Raise an exception for bad status codes

# Save the file locally
local_file_path = '/content/Auto_Theft_Open_Data.csv'
with open(local_file_path, 'wb') as f:
    f.write(response.content)

df = pd.read_csv(local_file_path, encoding="utf-8")

In [ ]:
len(df)

76749

In [ ]:
df.describe()
df.head()
print(df.columns)

Index(['OBJECTID', 'EVENT_UNIQUE_ID', 'REPORT_DATE', 'OCC_DATE', 'REPORT_YEAR',
       'REPORT_MONTH', 'REPORT_DAY', 'REPORT_DOY', 'REPORT_DOW', 'REPORT_HOUR',
       'OCC_YEAR', 'OCC_MONTH', 'OCC_DAY', 'OCC_DOY', 'OCC_DOW', 'OCC_HOUR',
       'DIVISION', 'LOCATION_TYPE', 'PREMISES_TYPE', 'UCR_CODE', 'UCR_EXT',
       'OFFENCE', 'CSI_CATEGORY', 'HOOD_158', 'NEIGHBOURHOOD_158', 'Region',
       'HOOD_140', 'NEIGHBOURHOOD_140', 'LONG_WGS84', 'LAT_WGS84', 'x', 'y'],
      dtype='object')


clean the hood_158 values because the value will be used for grouping and for determining the zone

In [ ]:
len(df)

76749

In [ ]:
df[~df['HOOD_158'].astype(str).str.isnumeric()]

,OBJECTID,EVENT_UNIQUE_ID,REPORT_DATE,OCC_DATE,REPORT_YEAR,REPORT_MONTH,REPORT_DAY,REPORT_DOY,REPORT_DOW,REPORT_HOUR,...,CSI_CATEGORY,HOOD_158,NEIGHBOURHOOD_158,Region,HOOD_140,NEIGHBOURHOOD_140,LONG_WGS84,LAT_WGS84,x,y
31,32,GO-20141295108,01/07/14 5:00,12/26/2013 5:00:00 AM,2014,January,7,7,Tuesday,0,...,Auto Theft,NSA,NSA,NaN,NSA,NSA,0.00000,0.000000,6.330000e-09,5.660000e-09
138,139,GO-20141380736,1/20/2014 5:00:00 AM,1/20/2014 5:00:00 AM,2014,January,20,20,Monday,20,...,Auto Theft,NSA,NSA,NaN,NSA,NSA,-79.54064,43.771526,-8.854424e+06,5.430153e+06
273,274,GO-20141478038,02/06/14 5:00,02/05/14 5:00,2014,February,6,37,Thursday,8,...,Auto Theft,NSA,NSA,NaN,NSA,NSA,0.00000,0.000000,6.330000e-09,5.660000e-09
286,287,GO-20141483961,02/07/14 5:00,02/06/14 5:00,2014,February,7,38,Friday,5,...,Auto Theft,NSA,NSA,NaN,NSA,NSA,0.00000,0.000000,6.330000e-09,5.660000e-09
322,323,GO-20141521796,2/13/2014 5:00:00 AM,02/12/14 5:00,2014,February,13,44,Thursday,11,...,Auto Theft,NSA,NSA,NaN,NSA,NSA,0.00000,0.000000,6.330000e-09,5.660000e-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76014,76015,GO-20252486085,11/28/2025 5:00:00 AM,11/28/2025 5:00:00 AM,2025,November,28,332,Friday,2,...,Auto Theft,NSA,NSA,NaN,NSA,NSA,0.00000,0.000000,6.330000e-09,5.660000e-09
76063,76064,GO-20252505724,11/30/2025 5:00:00 AM,11/30/2025 5:00:00 AM,2025,November,30,334,Sunday,20,...,Auto Theft,NSA,NSA,NaN,NSA,NSA,0.00000,0.000000,6.330000e-09,5.660000e-09
76243,76244,GO-20252556916,12/07/25 5:00,11/30/2025 5:00:00 AM,2025,December,7,341,Sunday,18,...,Auto Theft,NSA,NSA,NaN,NSA,NSA,0.00000,0.000000,6.330000e-09,5.660000e-09
76542,76543,GO-20252640438,12/20/2025 5:00:00 AM,12/16/2025 5:00:00 AM,2025,December,20,354,Saturday,1,...,Auto Theft,NSA,NSA,NaN,NSA,NSA,0.00000,0.000000,6.330000e-09,5.660000e-09


In [ ]:
df['HOOD_158'] = pd.to_numeric(df['HOOD_158'], errors='coerce')
df = df.dropna(subset=['HOOD_158'])     # remove rows where conversion failed
df['HOOD_158'] = df['HOOD_158'].astype(int)

In [ ]:
len(df)

75875

In [ ]:
df["PREMISES_TYPE"].unique()
df["PREMISES_TYPE"].value_counts()

,count
PREMISES_TYPE,
Outside,41237
House,23270
Commercial,6263
Apartment,2483
Other,2282
Transit,245
Educational,95


In [ ]:
len(df)

75875

In [ ]:
df["LOCATION_TYPE"].unique()
df["LOCATION_TYPE"].value_counts()

,count
LOCATION_TYPE,
"Parking Lots (Apt., Commercial Or Non-Commercial)",27100
"Single Home, House (Attach Garage, Cottage, Mobile)",23270
"Streets, Roads, Highways (Bicycle Path, Private Road)",13950
"Other Commercial / Corporate Places (For Profit, Warehouse, Corp. Bldg",3822
"Apartment (Rooming House, Condo)",2483
"Private Property Structure (Pool, Shed, Detached Garage)",1893
"Dealership (Car, Motorcycle, Marine, Trailer, Etc.)",1367
"Commercial Dwelling Unit (Hotel, Motel, B & B, Short Term Rental)",383
"Gas Station (Self, Full, Attached Convenience)",319


In [ ]:
len(df)

75875

In [ ]:
grouped = df.groupby(['PREMISES_TYPE', 'LOCATION_TYPE']).size().reset_index(name='COUNT')
print(grouped)

   PREMISES_TYPE                                      LOCATION_TYPE  COUNT
0      Apartment                   Apartment (Rooming House, Condo)   2483
1     Commercial  Bank And Other Financial Institutions (Money M...     48
2     Commercial                                   Bar / Restaurant    154
3     Commercial  Commercial Dwelling Unit (Hotel, Motel, B & B,...    383
4     Commercial       Construction Site (Warehouse, Trailer, Shed)    102
5     Commercial                                 Convenience Stores     68
6     Commercial  Dealership (Car, Motorcycle, Marine, Trailer, ...   1367
7     Commercial     Gas Station (Self, Full, Attached Convenience)    319
8     Commercial  Other Commercial / Corporate Places (For Profi...   3822
9    Educational                 Schools During Supervised Activity     26
10   Educational              Schools During Un-Supervised Activity     56
11   Educational                            Universities / Colleges     13
12         House  Single 

In [ ]:
len(df)

75875

Since there are too many LOCATION_TYPES, group them into broader categories.

In [ ]:
df_updated = df.copy()

# clean text first
df_updated["PREMISES_TYPE"] = df_updated["PREMISES_TYPE"].astype(str).str.strip()
df_updated["LOCATION_TYPE"] = df_updated["LOCATION_TYPE"].astype(str).str.strip()

# Apartment
df_updated.loc[df_updated["PREMISES_TYPE"] == "Apartment", "LOCATION_TYPE"] = "Apartment"

# Educational
df_updated.loc[df_updated["PREMISES_TYPE"] == "Educational", "LOCATION_TYPE"] = "Educational"

# Other
df_updated.loc[
    (df_updated["PREMISES_TYPE"] == "Other") &
    (df_updated["LOCATION_TYPE"].str.contains("Private", case=False, na=False)),
    "LOCATION_TYPE"
] = "Private"

df_updated.loc[
    (df_updated["PREMISES_TYPE"] == "Other") &
    (~df_updated["LOCATION_TYPE"].str.contains("Private", case=False, na=False)),
    "LOCATION_TYPE"
] = "Public"

# Outside
df_updated.loc[
    (df_updated["PREMISES_TYPE"] == "Outside") &
    (df_updated["LOCATION_TYPE"].str.contains("Parking Lots", case=False, na=False)),
    "LOCATION_TYPE"
] = "Parking"

df_updated.loc[
    (df_updated["PREMISES_TYPE"] == "Outside") &
    (df_updated["LOCATION_TYPE"].str.contains("Streets", case=False, na=False)),
    "LOCATION_TYPE"
] = "Street"

df_updated.loc[
    (df_updated["PREMISES_TYPE"] == "Outside") &
    (~df_updated["LOCATION_TYPE"].str.contains("Parking Lots", case=False, na=False)) &
    (~df_updated["LOCATION_TYPE"].str.contains("Streets", case=False, na=False)),
    "LOCATION_TYPE"
] = "Others"

# Transit
df_updated.loc[
    (df_updated["PREMISES_TYPE"] == "Transit") &
    (df_updated["LOCATION_TYPE"].str.contains("Ttc", case=False, na=False)),
    "LOCATION_TYPE"
] = "TTC"

df_updated.loc[
    (df_updated["PREMISES_TYPE"] == "Transit") &
    (~df_updated["LOCATION_TYPE"].str.contains("Ttc", case=False, na=False)),
    "LOCATION_TYPE"
] = "Others"

# House
df_updated.loc[df_updated["PREMISES_TYPE"] == "House", "LOCATION_TYPE"] = "House"

# Commercial
df_updated.loc[
    (df_updated["PREMISES_TYPE"] == "Commercial") &
    (df_updated["LOCATION_TYPE"].str.contains("Other Commercial", case=False, na=False)),
    "LOCATION_TYPE"
] = "Others"

df_updated.loc[
    (df_updated["PREMISES_TYPE"] == "Commercial") &
    (~df_updated["LOCATION_TYPE"].str.contains("Other Commercial", case=False, na=False)),
    "LOCATION_TYPE"
] = "Commercial"

df = df_updated.copy()

In [ ]:
len(df)

75875

In [ ]:
grouped = df.groupby(['PREMISES_TYPE', 'LOCATION_TYPE']).size().reset_index(name='COUNT')
print(grouped)

  PREMISES_TYPE LOCATION_TYPE  COUNT
0     Apartment     Apartment   2483
1    Commercial    Commercial   6263
2   Educational   Educational     95
3         House         House  23270
4         Other       Private   1893
5         Other        Public    389
6       Outside        Others  41237
7       Transit        Others    112
8       Transit           TTC    133


group by hood_58, report_year, report_month, premise_type, location_type to create the count column containing the total thefts for that period

In [ ]:
grouped = (
    df_updated
    .groupby(['HOOD_158', 'REPORT_YEAR', 'REPORT_MONTH', 'PREMISES_TYPE', 'LOCATION_TYPE'])
    .size()
    .reset_index(name='COUNT')
)

print(grouped)

       HOOD_158  REPORT_YEAR REPORT_MONTH PREMISES_TYPE LOCATION_TYPE  COUNT
0             1         2014        April    Commercial    Commercial      6
1             1         2014        April         House         House      1
2             1         2014        April         Other        Public      1
3             1         2014        April       Outside        Others     19
4             1         2014        April       Transit        Others      1
...         ...          ...          ...           ...           ...    ...
31879       174         2025        March       Outside        Others      1
31880       174         2025     November         House         House      1
31881       174         2025     November       Outside        Others      1
31882       174         2025      October       Outside        Others      1
31883       174         2025    September       Outside        Others      1

[31884 rows x 6 columns]


In [ ]:
len(grouped)

31884

Create additional columns

* PREV_12_MONTHS_COUNT
* PREV_6_MONTHS_COUNT
* PREV_1_MONTH_COUNT
* NEXT_1_MONTH_COUNT
* NEXT_6_MONTHS_COUNT
* NEXT_12_MONTHS_COUNT

I assume that that PREV_* and NEXT_* counts give a good overview of the thefts relative to the current month.

In [ ]:
# original df
df = grouped.copy()

# 1. create DATE in original df
df['DATE'] = pd.to_datetime(
    df['REPORT_YEAR'].astype(str) + '-' +
    df['REPORT_MONTH'].astype(str) + '-01'
)

group_cols = ['HOOD_158', 'PREMISES_TYPE', 'LOCATION_TYPE']

# 2. create a helper monthly table: one row per hood-premise-location-month
monthly_helper = (
    df.groupby(group_cols + ['DATE'], as_index=False)['COUNT']
    .sum()
    .sort_values(group_cols + ['DATE'])
)

# 3. compute prev/next on helper table only
monthly_helper['PREV_1_MONTH_COUNT'] = (
    monthly_helper.groupby(group_cols)['COUNT'].shift(1)
)

monthly_helper['PREV_6_MONTHS_COUNT'] = (
    monthly_helper.groupby(group_cols)['COUNT']
    .transform(lambda x: x.shift(1).rolling(6, min_periods=1).sum())
)

monthly_helper['PREV_12_MONTHS_COUNT'] = (
    monthly_helper.groupby(group_cols)['COUNT']
    .transform(lambda x: x.shift(1).rolling(12, min_periods=1).sum())
)

monthly_helper['NEXT_1_MONTH_COUNT'] = (
    monthly_helper.groupby(group_cols)['COUNT'].shift(-1)
)

monthly_helper['NEXT_6_MONTHS_COUNT'] = (
    monthly_helper.groupby(group_cols)['COUNT']
    .transform(lambda x: x[::-1].shift(1).rolling(6, min_periods=1).sum()[::-1])
)

monthly_helper['NEXT_12_MONTHS_COUNT'] = (
    monthly_helper.groupby(group_cols)['COUNT']
    .transform(lambda x: x[::-1].shift(1).rolling(12, min_periods=1).sum()[::-1])
)

# 4. merge the new columns back to original df
df = df.merge(
    monthly_helper[
        group_cols + ['DATE',
        'PREV_1_MONTH_COUNT',
        'PREV_6_MONTHS_COUNT',
        'PREV_12_MONTHS_COUNT',
        'NEXT_1_MONTH_COUNT',
        'NEXT_6_MONTHS_COUNT',
        'NEXT_12_MONTHS_COUNT']
    ],
    on=group_cols + ['DATE'],
    how='left'
)

print(df.shape)   # should stay same as original grouped.shape
print(df.head(15))

(31884, 13)
    HOOD_158  REPORT_YEAR REPORT_MONTH PREMISES_TYPE LOCATION_TYPE  COUNT  \
0          1         2014        April    Commercial    Commercial      6   
1          1         2014        April         House         House      1   
2          1         2014        April         Other        Public      1   
3          1         2014        April       Outside        Others     19   
4          1         2014        April       Transit        Others      1   
5          1         2014       August    Commercial    Commercial      7   
6          1         2014       August         House         House      4   
7          1         2014       August       Outside        Others     34   
8          1         2014     December    Commercial    Commercial      7   
9          1         2014     December         House         House      3   
10         1         2014     December         Other       Private      1   
11         1         2014     December         Other        Publ

In [ ]:
len(df)

31884

Drop PREV_* and NEXT_* rows with no values. These are rows from the beginning and the next of the data set.

i.e. the first few row in the data will of course not have prev_* data, while the last few rows will not have NEXT_* data


Drop the column DATE which was created when the PREV_* and NEXT_* were created.

In [ ]:
cols = [
    'PREV_1_MONTH_COUNT',
    'PREV_6_MONTHS_COUNT',
    'PREV_12_MONTHS_COUNT',
    'NEXT_1_MONTH_COUNT',
    'NEXT_6_MONTHS_COUNT',
    'NEXT_12_MONTHS_COUNT'
]

monthly_clean = df.dropna(subset=cols)
monthly_clean = monthly_clean.drop(columns=['DATE'])
df = monthly_clean.copy()

In [ ]:
len(df)

30026

In [ ]:
print(df.head(15))

    HOOD_158  REPORT_YEAR REPORT_MONTH PREMISES_TYPE LOCATION_TYPE  COUNT  \
0          1         2014        April    Commercial    Commercial      6   
1          1         2014        April         House         House      1   
3          1         2014        April       Outside        Others     19   
5          1         2014       August    Commercial    Commercial      7   
6          1         2014       August         House         House      4   
7          1         2014       August       Outside        Others     34   
8          1         2014     December    Commercial    Commercial      7   
9          1         2014     December         House         House      3   
11         1         2014     December         Other        Public      1   
12         1         2014     December       Outside        Others     14   
13         1         2014     December       Transit        Others      1   
14         1         2014     February    Commercial    Commercial      6   

In [ ]:
df.describe()

,HOOD_158,REPORT_YEAR,COUNT,PREV_1_MONTH_COUNT,PREV_6_MONTHS_COUNT,PREV_12_MONTHS_COUNT,NEXT_1_MONTH_COUNT,NEXT_6_MONTHS_COUNT,NEXT_12_MONTHS_COUNT
count,30026.000000,30026.000000,30026.000000,30026.000000,30026.000000,30026.000000,30026.000000,30026.000000,30026.000000
mean,83.757543,2020.278026,2.437954,2.424132,13.924332,26.500067,2.449444,14.193099,27.308533
std,53.334187,3.248749,2.775766,2.766492,14.440396,28.357021,2.783368,14.458859,28.400243
min,1.000000,2014.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,33.000000,2018.000000,1.000000,1.000000,8.000000,15.000000,1.000000,8.000000,16.000000
50%,87.000000,2021.000000,2.000000,2.000000,11.000000,21.000000,2.000000,11.000000,22.000000
75%,130.000000,2023.000000,3.000000,3.000000,16.000000,31.000000,3.000000,16.000000,32.000000
max,174.000000,2025.000000,68.000000,68.000000,338.000000,608.000000,68.000000,338.000000,608.000000


In [ ]:
df[df['PREV_12_MONTHS_COUNT'] == 0]

,HOOD_158,REPORT_YEAR,REPORT_MONTH,PREMISES_TYPE,LOCATION_TYPE,COUNT,PREV_1_MONTH_COUNT,PREV_6_MONTHS_COUNT,PREV_12_MONTHS_COUNT,NEXT_1_MONTH_COUNT,NEXT_6_MONTHS_COUNT,NEXT_12_MONTHS_COUNT


Drop REPORT_YEAR and REPORT_MONTH as columns.

This will make the data nominal.

Also, since issuing insurance is prospective, month and especially year are irrelevant as features to determine if the next few months will be high risk or not.

In [ ]:
df = df.rename(columns={
    'COUNT': 'CURRENT_MONTH'
}).drop(columns=['REPORT_YEAR', 'REPORT_MONTH'], errors='ignore')

In [ ]:
#median_val = df['NEXT_12_MONTHS_COUNT'].median()
#print(median_val)

In [ ]:
'''df['IS_HIGH_RISK'] = (
    df['NEXT_12_MONTHS_COUNT'] < median_val
).astype(int)'''

"df['IS_HIGH_RISK'] = (\n    df['NEXT_12_MONTHS_COUNT'] < median_val\n).astype(int)"

In [ ]:
len(df)

30026

In [ ]:
#df['IS_HIGH_RISK'].value_counts()

In [ ]:
df.head()

,HOOD_158,PREMISES_TYPE,LOCATION_TYPE,CURRENT_MONTH,PREV_1_MONTH_COUNT,PREV_6_MONTHS_COUNT,PREV_12_MONTHS_COUNT,NEXT_1_MONTH_COUNT,NEXT_6_MONTHS_COUNT,NEXT_12_MONTHS_COUNT
0,1,Commercial,Commercial,6,7.0,15.0,15.0,3.0,17.0,48.0
1,1,House,House,1,4.0,10.0,10.0,2.0,23.0,39.0
3,1,Outside,Others,19,16.0,36.0,36.0,28.0,133.0,227.0
5,1,Commercial,Commercial,7,2.0,25.0,27.0,2.0,28.0,49.0
6,1,House,House,4,2.0,13.0,13.0,5.0,21.0,42.0


one-hot encoding

In [ ]:
df = pd.get_dummies(df, columns=["HOOD_158","PREMISES_TYPE","LOCATION_TYPE"])

In [ ]:
list(df.columns)

['CURRENT_MONTH',
 'PREV_1_MONTH_COUNT',
 'PREV_6_MONTHS_COUNT',
 'PREV_12_MONTHS_COUNT',
 'NEXT_1_MONTH_COUNT',
 'NEXT_6_MONTHS_COUNT',
 'NEXT_12_MONTHS_COUNT',
 'HOOD_158_1',
 'HOOD_158_2',
 'HOOD_158_3',
 'HOOD_158_4',
 'HOOD_158_5',
 'HOOD_158_6',
 'HOOD_158_7',
 'HOOD_158_8',
 'HOOD_158_9',
 'HOOD_158_10',
 'HOOD_158_11',
 'HOOD_158_12',
 'HOOD_158_13',
 'HOOD_158_15',
 'HOOD_158_16',
 'HOOD_158_18',
 'HOOD_158_19',
 'HOOD_158_20',
 'HOOD_158_21',
 'HOOD_158_22',
 'HOOD_158_23',
 'HOOD_158_24',
 'HOOD_158_25',
 'HOOD_158_27',
 'HOOD_158_28',
 'HOOD_158_29',
 'HOOD_158_30',
 'HOOD_158_31',
 'HOOD_158_32',
 'HOOD_158_33',
 'HOOD_158_34',
 'HOOD_158_35',
 'HOOD_158_36',
 'HOOD_158_37',
 'HOOD_158_38',
 'HOOD_158_39',
 'HOOD_158_40',
 'HOOD_158_41',
 'HOOD_158_42',
 'HOOD_158_43',
 'HOOD_158_44',
 'HOOD_158_46',
 'HOOD_158_47',
 'HOOD_158_48',
 'HOOD_158_49',
 'HOOD_158_50',
 'HOOD_158_52',
 'HOOD_158_53',
 'HOOD_158_54',
 'HOOD_158_55',
 'HOOD_158_56',
 'HOOD_158_57',
 'HOOD_158_58',

Saving the new file to a file

In [ ]:
df.to_csv("cleaned_data.csv", index=False)

# MODELLING

loading the data

In [ ]:
df = pd.read_csv("/content/cleaned_data.csv", encoding="utf-8")

Splitting the data

In [ ]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(df, test_size=0.3, random_state=42)

In [ ]:
target = 'NEXT_12_MONTHS_COUNT'
features = list(train_set.columns)
features = [f for f in features if f!=target]

In [ ]:
X_tr = train_set[features]
y_tr = train_set[[target]]

X_te = test_set[features]
y_te = test_set[[target]]

Scaling the data

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_tr)

X_tr = scaler.transform(X_tr)
X_te = scaler.transform(X_te)

##Regression

We would like to predict the number of thefts for the next 12 months.

**LinearRegression**

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
import numpy as np

def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())

In [ ]:
from sklearn.linear_model import LinearRegression

lin_scores = cross_val_score(LinearRegression(), train_set[features], train_set[target], scoring="neg_mean_squared_error", cv=4)

lin_rmse_scores = np.sqrt(-lin_scores)
display_scores(lin_rmse_scores)


Scores: [7.46314243 7.48536099 7.67664418 7.58945357]
Mean: 7.553650292068994


**Logistic Regression**

In [ ]:
df[['NEXT_12_MONTHS_COUNT']].describe()

,NEXT_12_MONTHS_COUNT
count,30026.000000
mean,27.308533
std,28.400243
min,1.000000
25%,16.000000
50%,22.000000
75%,32.000000
max,608.000000


In [ ]:
bins = [0, 5, 10, 20, 50, 100, df['NEXT_12_MONTHS_COUNT'].max()]

df['RANGE'] = pd.cut(df['NEXT_12_MONTHS_COUNT'], bins=bins)

df['RANGE'].value_counts().sort_index()

,count
RANGE,
"(0.0, 5.0]",2344
"(5.0, 10.0]",1815
"(10.0, 20.0]",9242
"(20.0, 50.0]",14098
"(50.0, 100.0]",2027
"(100.0, 608.0]",500


In [ ]:
total = len(df)
count_above = (df['NEXT_12_MONTHS_COUNT'] > 32).sum()
percentage = count_above / total * 100

count_above, percentage

(np.int64(7077), np.float64(23.569573036701524))

In [ ]:
y_tr_b = 1*np.ravel(y_tr>=median_theft_count)
y_te_b = 1*np.ravel(y_te>=median_theft_count)

In [ ]:
from sklearn.linear_model import LogisticRegression
import pandas as pd

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_tr, y_tr_b.squeeze())

coef = pd.Series(log_model.coef_[0], index=features)

print(coef.sort_values(ascending=False))

NEXT_6_MONTHS_COUNT      8.378536
PREV_6_MONTHS_COUNT      0.950265
PREMISES_TYPE_Outside    0.238661
LOCATION_TYPE_Others     0.195938
CURRENT_MONTH            0.182128
                           ...   
HOOD_158_69             -0.520397
HOOD_158_140            -0.529882
PREMISES_TYPE_Transit   -0.551055
PREV_12_MONTHS_COUNT    -0.625498
HOOD_158_79             -0.691623
Length: 179, dtype: float64


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# predict class (0 or 1)
y_pred = log_model.predict(X_te)

# accuracy
print("Accuracy:", accuracy_score(y_te_b, y_pred))

# confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_te_b, y_pred))

# full report
print("\nClassification Report:")
print(classification_report(y_te_b, y_pred))

Accuracy: 0.8762211367673179

Confusion Matrix:
[[3852  506]
 [ 609 4041]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.88      0.87      4358
           1       0.89      0.87      0.88      4650

    accuracy                           0.88      9008
   macro avg       0.88      0.88      0.88      9008
weighted avg       0.88      0.88      0.88      9008



In [ ]:
import pandas as pd
import numpy as np

# get trained logistic regression from pipeline
log_reg = model.named_steps["log_reg"]

# get feature names
feature_names = train_set[features].columns

# get coefficients
coefficients = log_reg.coef_[0]

# create dataframe for easy viewing
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Abs_Impact": np.abs(coefficients)
})

# sort by impact
coef_df = coef_df.sort_values(by="Abs_Impact", ascending=False)

print(coef_df)

NameError: name 'model' is not defined

##Classification

We would like to predict if next 12 months is high risk or low risk for for theft.

Creating another yhat which now represents if the next month is high risk or low risk for auto theft. The basis will be the median value of NEXT_12_MONTHS_COUNT. Count that is higher than the median is considered as high risk, otherwise, it is low risk.

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
median_theft_count = np.median(df[['NEXT_12_MONTHS_COUNT']])

In [ ]:
y_tr_b = 1*np.ravel(y_tr>=median_theft_count)
y_te_b = 1*np.ravel(y_te>=median_theft_count)

**Linear SVC**

In [ ]:
from sklearn.svm import LinearSVC

lin_clf = LinearSVC(random_state=42)
lin_clf.fit(X_tr, y_tr_b)

y_pred = lin_clf.predict(X_te)
accuracy_score(y_te_b, y_pred)

0.8763321492007105

**SVC**

In [ ]:
from sklearn.svm import SVC

In [ ]:
svc_clf = SVC(random_state=42, C=1.0, gamma='scale')
svc_clf.fit(X_tr, y_tr_b)
y_pred_svc = svc_clf.predict(X_te)
accuracy_score(y_te_b, y_pred_svc)

lin_clf = SVC(random_state=42)
lin_clf.fit(X_tr, y_tr_b)

y_pred = lin_clf.predict(X_tr)
accuracy_score(y_tr_b, y_pred)

0.8557426967361309

### Randomized CV

In [ ]:
'''
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import reciprocal, uniform

param_distributions = {
    'C': uniform(300, 500),
    'gamma': reciprocal(0.01, 0.1)
}

rnd_search = RandomizedSearchCV(
    SVC(random_state=42),
    param_distributions=param_distributions,
    n_iter=10,
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

rnd_search.fit(X_tr, y_tr_b)

rnd_search.best_params_
rnd_search.best_score_'''